> Tensorflow Documentation - https://www.tensorflow.org/api_docs/python/

UV is An extremely fast Python package and project manager, faster 10-100x time than pip.
Installing UV, setting Up Virtual Environment and downloading depending dependencies.
```sh
pip install uv
uv venv
uv pip install -r requirements.txt
```

In [ ]:
import tensorflow as tf # models
import numpy as np # math computation
import matplotlib.pyplot as plt # plotting charts
import tensorflow_datasets as tfds # to load dataset
from tensorflow.keras.models import Model
from tensorflow.keras.layers import InputLayer, Conv2D, MaxPool2D, Dense, Flatten, BatchNormalization
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import Adam

# Convolutional Neural Network - CNN [Malaria Diagnosis]

## Loading Dataset

This line of code uses `tfds.load` to retrieve the 'malaria' dataset. `with_info = True` ensures that dataset_info contains metadata about the dataset, while `as_supervised = True` formats the data as image-label pairs, suitable for supervised learning. `shuffle_files = True` shuffles the order of data files to prevent bias, and `split = ['train']` specifically loads only the training portion of the dataset, preparing it for model training.

In [ ]:
dataset, dataset_info = tfds.load('malaria', with_info = True, as_supervised = True, shuffle_files = True, split = ['train'])

## Understanding Dataset

In [ ]:
dataset

In [ ]:
dataset_info

In [ ]:
# take() tells us how manu values to display
for data in dataset[0].take(4):
  print(data)

## Splitting Dataset

In [ ]:
def splits(dataset, TRAIN_RATIO, VAL_RATIO, TEST_RATIO):
    DATASET_SIZE = len(dataset)

    train_dataset = dataset.take(int(TRAIN_RATIO*DATASET_SIZE))

    # skip() will skip the elements in () and start display after it
    val_test_dataset = dataset.skip(int(TRAIN_RATIO*DATASET_SIZE))
    val_dataset = val_test_dataset.take(int(VAL_RATIO*DATASET_SIZE))
    test_dataset = val_test_dataset.skip(int(VAL_RATIO*DATASET_SIZE))

    return train_dataset, val_dataset, test_dataset

In [ ]:
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

train_dataset, val_dataset, test_dataset = splits(dataset[0], TRAIN_RATIO, VAL_RATIO, TEST_RATIO)
print( [int(x[1]) for x in train_dataset.as_numpy_iterator()] )
print( [int(x[1]) for x in val_dataset.as_numpy_iterator()] )
print( [int(x[1]) for x in test_dataset.as_numpy_iterator()] )

## Data Visualization

In [ ]:
dataset_info

In [ ]:
dataset_info.features['label'].int2str(0)

In [ ]:
dataset_info.features['label'].int2str(1)

In [ ]:
for i, (image, label) in enumerate(train_dataset.take(16)):
  ax = plt.subplot(4, 4, i +1)
  plt.imshow(image)
  plt.title(dataset_info.features['label'].int2str(label))
  plt.axis('off')

## Data Preprocessing

### Resizing & Normalization

X = (X - Xmin) / (X max - X min) = (X - 0)/(255-0) = X/255

In [ ]:
IM_SIZE = 224
def resize_rescale(image, label):
  return tf.image.resize(image, (IM_SIZE, IM_SIZE))/255.0, label

In [ ]:
train_dataset = train_dataset.map(resize_rescale)
val_dataset = val_dataset.map(resize_rescale)
test_dataset = test_dataset.map(resize_rescale)

for image, label in train_dataset.take(1):
  print(image, label)

In [ ]:
BATCH_SIZE = 32
train_dataset = train_dataset.shuffle(buffer_size = 8, reshuffle_each_iteration = True).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.shuffle(buffer_size = 8, reshuffle_each_iteration = True).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

## Designing Covnets (Convolutional Neural Network)

- **CNN Explainer** - https://poloclub.github.io/cnn-explainer/

- `Wo = Wi - ((F + 2 * P) / S) + 1`
where
  - Wo = Weight of output
  - Wi = Weight of input
  - F = Field of Size (Kernel Size)
  - P = Padding
  - S = Stride

- **Explained Visually | Setosa** - https://setosa.io/ev/image-kernels/
- **LeNet** - https://en.wikipedia.org/wiki/LeNet
- **Demo on MNIST** - https://cs.stanford.edu/people/karpathy/convnetjs/demo/mnist.html
- **Conv2D** - https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D
- **MaxPool2D** - https://www.tensorflow.org/api_docs/python/tf/keras/layers/MaxPool2D

In [ ]:
lenet_model = tf.keras.Sequential([
    InputLayer(input_shape =(IM_SIZE, IM_SIZE, 3)),
    Conv2D(filters = 6, kernel_size = 3, strides=1, padding='valid', activation = 'relu'),
    BatchNormalization(),
    MaxPool2D(pool_size = 2, strides = 2),

    Conv2D(filters = 16, kernel_size = 3, strides=1, padding='valid', activation = 'relu'),
    BatchNormalization(),
    MaxPool2D(pool_size = 2, strides = 2),

    Flatten(),
    Dense(1000, activation = 'relu'),
    BatchNormalization(),

    Dense(100, activation = 'relu'),
    BatchNormalization(),

    Dense(1, activation = 'sigmoid')
])
lenet_model.summary()

### Binary Cross Entropy Loss

- **Loss Functions** - https://ml-cheatsheet.readthedocs.io/en/latest/loss_functions.html

In [ ]:
y_true = [0,1,0,0]
y_pred = [0.6,0.51,0.94,0]
bce = tf.keras.losses.BinaryCrossentropy()
bce(y_true, y_pred)

In [ ]:
lenet_model.compile(optimizer = Adam(learning_rate = 0.01),
                    loss = BinaryCrossentropy(),
                    metrics = ['accuracy'] )

In [ ]:
history = lenet_model.fit(train_dataset, validation_data = val_dataset, epochs = 20, verbose = 1)

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train_loss', 'val_loss'])
plt.show()

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['train_accuracy', 'val_accuracy'])
plt.show()

## Model Evaluation & Testing

In [ ]:
test_dataset = test_dataset.batch(1)
lenet_model.evaluate(test_dataset)

In [ ]:
def parasite_or_not(x):
  if(x<0.5):
    return str('P') # Parasitized cell
  else:
    return str('U') # Uninfected cell

In [ ]:
parasite_or_not(lenet_model.predict(test_dataset.take(1))[0][0])

In [ ]:
for i, (image, label) in enumerate(test_dataset.take(9)):
  ax = plt.subplot(3, 3, i+1)
  plt.imshow(image[0])
  plt.title(str(parasite_or_not(label.numpy()[0])) + ":"+ str(parasite_or_not(lenet_model.predict(image)[0][0])))

  plt.axis('off')

## Corrective Measures - To Prevent Overfitting

1.   Hyperparameter Tuning
2.   Smaller Network
3.   Early Stopping
4.   Regularization
5.   Dropout
6.   Collect More Data (Representative & Diverse)
7.   Data Augmentation
